In [48]:
"""SPLADE sparse embedding client using Triton Inference Server."""

from __future__ import annotations

from typing import Any, cast
import pickle
import numpy as np
from loguru import logger
from pydantic import BaseModel
from qdrant_client.models import SparseVector
from tritonclient.grpc import InferenceServerClient, InferInput, InferRequestedOutput, InferResult 


class SpladeConfig(BaseModel):
    """Configuration for SPLADE Triton client."""

    url: str = "localhost:8001"
    model_name: str = "splade"
    timeout: float = 30
    verbose: bool = False


class SpladeClient:
    def __init__(self, config: SpladeConfig):
        self.config = config
        self._client: InferenceServerClient | None = None

    def _get_client(self) -> InferenceServerClient:
        if self._client is None:
            self._client = InferenceServerClient(
                url=self.config.url,
                verbose=self.config.verbose,
            )
        return self._client

    def close(self) -> None:
        if self._client is not None:
            self._client.close()
            self._client = None

    def __enter__(self) -> "SpladeClient":
        return self

    def __exit__(self, exc_type: Any, exc_val: Any, exc_tb: Any) -> None:
        self.close()

    def check_health(self) -> bool:
        try:
            client = self._get_client()

            if not client.is_server_live():
                logger.error("[SpladeClient] Triton server is not live")
                return False

            if not client.is_model_ready(self.config.model_name):
                logger.error(
                    f"[SpladeClient] Model '{self.config.model_name}' is not ready"
                )
                return False

            logger.success(
                f"[SpladeClient] Triton server healthy, model '{self.config.model_name}' ready"
            )
            return True

        except Exception as e:
            logger.error(f"[SpladeClient] Health check failed: {e}")
            return False

    def encode(self, texts: list[str]) -> list[SparseVector]:
        if not texts:
            return []

        client = self._get_client()

    
        text_input = InferInput("TEXT", [len(texts), 1], "BYTES")
        text_input.set_data_from_numpy(np.array([[t] for t in texts], dtype=object))

        outputs = [
            InferRequestedOutput("INDICES"),
            InferRequestedOutput("VALUES"),
        ]

        try:
            result = client.infer(
                model_name=self.config.model_name,
                inputs=[text_input],
                outputs=outputs,
                timeout=self.config.timeout,
            )
        except Exception as e:
            logger.error(f"[SpladeClient] Inference failed: {e}")
            raise RuntimeError(f"SPLADE inference failed: {e}") from e

        if result is None:
            logger.error(f"[SpladeClient] Inference failed: result return is None for some reason")
            raise RuntimeError(f"SPLADE inference failed. Result return None")
        
        
        indices_batch = result.as_numpy("INDICES")
        values_batch = result.as_numpy("VALUES")
        
        print(indices_batch)
        
        assert indices_batch is not None, "[SpladeClient] Indices batch is None for some reason"
        assert values_batch is not None, "[SpladeClient] Value batch is None for some reason"

        sparse_vectors = []
        for i in range(len(texts)):
            indices = pickle.loads(indices_batch[i]).tolist()
            values = pickle.loads(values_batch[i]).tolist()

            sparse_vectors.append(
                SparseVector(
                    indices=indices,
                    values=values,
                )
            )

        logger.debug(
            f"[SpladeClient] Encoded {len(texts)} text(s) to sparse vectors"
        )
        return sparse_vectors

    async def aencode(self, texts: list[str]) -> list[SparseVector]:
        return self.encode(texts)



In [55]:
texts = [
    "This is a tes. -- @@3#@!#@!/",
    "Hello, world!",
]
client = SpladeClient(SpladeConfig())

In [56]:
client.encode(texts)

2026-03-16 13:23:36.591 | DEBUG    | __main__:encode:121 - [SpladeClient] Encoded 2 text(s) to sparse vectors


[b'\x80\x04\x95\xd1\x00\x00\x00\x00\x00\x00\x00\x8c\x16numpy._core.multiarray\x94\x8c\x0c_reconstruct\x94\x93\x94\x8c\x05numpy\x94\x8c\x07ndarray\x94\x93\x94K\x00\x85\x94C\x01b\x94\x87\x94R\x94(K\x01K\x12\x85\x94h\x03\x8c\x05dtype\x94\x93\x94\x8c\x02i4\x94\x89\x88\x87\x94R\x94(K\x03\x8c\x01<\x94NNNJ\xff\xff\xff\xffJ\xff\xff\xff\xffK\x00t\x94b\x89CH\xe7\x03\x00\x00\xe9\x03\x00\x00\xf5\x03\x00\x00\xf9\x03\x00\x00\x06\x04\x00\x00\r\x04\x00\x00\x11\x04\x00\x00 \x04\x00\x00\xd3\x07\x00\x00\xdf\x07\x00\x00\xe7\x07\x00\x00-\x08\x00\x00\xc1\x10\x00\x00R\x11\x00\x00\xd8\x13\x00\x00\xd3"\x00\x00\xf6G\x00\x00\xe2W\x00\x00\x94t\x94b.'
 b'\x80\x04\x95\xd9\x00\x00\x00\x00\x00\x00\x00\x8c\x16numpy._core.multiarray\x94\x8c\x0c_reconstruct\x94\x93\x94\x8c\x05numpy\x94\x8c\x07ndarray\x94\x93\x94K\x00\x85\x94C\x01b\x94\x87\x94R\x94(K\x01K\x14\x85\x94h\x03\x8c\x05dtype\x94\x93\x94\x8c\x02i4\x94\x89\x88\x87\x94R\x94(K\x03\x8c\x01<\x94NNNJ\xff\xff\xff\xffJ\xff\xff\xff\xffK\x00t\x94b\x89CP\xe7\x03\x00\x00\xe

[SparseVector(indices=[999, 1001, 1013, 1017, 1030, 1037, 1041, 1056, 2003, 2015, 2023, 2093, 4289, 4434, 5080, 8915, 18422, 22498], values=[0.40453484654426575, 1.173599362373352, 0.4684564471244812, 1.0309802293777466, 0.9332277774810791, 0.8521087169647217, 0.6147517561912537, 0.528121829032898, 0.46243035793304443, 1.971237301826477, 0.9627698063850403, 0.14728900790214539, 0.09152562916278839, 0.011685965582728386, 0.04627971351146698, 2.7071475982666016, 0.15046006441116333, 0.1321127861738205]),
 SparseVector(indices=[999, 2017, 2026, 2040, 2057, 2088, 2272, 2299, 2477, 2748, 3376, 3795, 4774, 5304, 6160, 6919, 7592, 7632, 8404, 8484], values=[1.9145121574401855, 0.0618186891078949, 0.09442538768053055, 0.35482174158096313, 0.0037486536893993616, 2.8006772994995117, 0.06492841988801956, 0.40679997205734253, 0.04372887685894966, 1.524186134338379, 0.19160765409469604, 0.8574978113174438, 0.22511076927185059, 0.08072520047426224, 1.1970536708831787, 0.1323975920677185, 2.726323604

In [57]:

texts[0].encode('utf-8')


b'This is a tes. -- @@3#@!#@!/'